# Walkthrough: from ECB curve parameters to what the market prices

Run `erm fetch` first. Every cell reads the local cache in `data/raw/`; nothing here
touches the network. Each step is a check you could be asked to do on a whiteboard.

In [1]:
import numpy as np
import pandas as pd

from euro_rates_monitor import build, curve, forwards, load, pca

spot = load.ecb_curve("ecb_spot_aaa")
params = load.ecb_wide("ecb_params_aaa").dropna()
day = spot.index[-1]
spot.tail(3).round(3)

,0.25,0.50,1.00,2.00,3.00,5.00,7.00,10.00,15.00,20.00,30.00
2026-09-24,2.612,2.793,3.030,3.229,3.288,3.343,3.425,3.566,3.742,3.812,3.744
2026-09-25,2.556,2.761,3.028,3.251,3.316,3.378,3.465,3.615,3.796,3.866,3.787
2026-09-28,2.563,2.756,3.011,3.233,3.305,3.379,3.473,3.631,3.825,3.906,3.843


## 1. The published curve is a Svensson fit, and we can reproduce it exactly

The ECB publishes six parameters per day. Evaluating the Svensson formula at the
published maturities must return the published spot rates.

In [2]:
ours = curve.spot_from_params(params.loc[day], np.array(spot.columns))
pd.DataFrame({"published": spot.loc[day], "from parameters": ours,
              "diff (bp)": (ours - spot.loc[day]) * 100}).round(6)

,published,from parameters,diff (bp)
0.25,2.563355,2.563355,0.0
0.50,2.756095,2.756095,0.0
1.00,3.011000,3.011000,0.0
2.00,3.233030,3.233030,0.0
3.00,3.304964,3.304964,0.0
5.00,3.378758,3.378758,0.0
7.00,3.473481,3.473481,0.0
10.00,3.631319,3.631319,0.0
15.00,3.825125,3.825125,0.0
20.00,3.906298,3.906298,0.0


## 2. One forward rate by hand

Continuous compounding: f(t1, t2) = (r2·t2 − r1·t1) / (t2 − t1). The 1Y rate in 1Y
from the 1Y and 2Y spot rates:

In [3]:
r1, r2 = spot.loc[day, 1.0], spot.loc[day, 2.0]
by_hand = (r2 * 2 - r1 * 1) / (2 - 1)
print(f"1Y spot {r1:.4f}%, 2Y spot {r2:.4f}%  ->  1Y1Y forward {by_hand:.4f}%")
print(f"library: {float(forwards.forward_rate(r1, 1.0, r2, 2.0)):.4f}%")

1Y spot 3.0110%, 2Y spot 3.2330%  ->  1Y1Y forward 3.4551%
library: 3.4551%


## 3. Short forwards converge to the instantaneous forward

The ECB technical note gives the instantaneous forward in closed form:
f(t) = b0 + b1·e^(−t/τ1) + b2·(t/τ1)·e^(−t/τ1) + b3·(t/τ2)·e^(−t/τ2).
As the forward period shrinks, our discrete forwards must approach it.

In [4]:
p = params.loc[day]
def inst_fwd(t):
    return (p.BETA0 + p.BETA1 * np.exp(-t / p.TAU1) + p.BETA2 * (t / p.TAU1) * np.exp(-t / p.TAU1)
            + p.BETA3 * (t / p.TAU2) * np.exp(-t / p.TAU2))
t = 1.0
for dt in (0.25, 0.05, 0.001):
    r_a, r_b = curve.spot_from_params(p, np.array([t, t + dt]))
    print(f"period {dt:>5}: discrete {float(forwards.forward_rate(r_a, t, r_b, t + dt)):.5f}%"
          f"   instantaneous {inst_fwd(t):.5f}%")

period  0.25: discrete 3.42010%   instantaneous 3.38832%
period  0.05: discrete 3.39589%   instantaneous 3.38832%
period 0.001: discrete 3.38848%   instantaneous 3.38832%


## 4. The path and its sensitivity to a term premium

Headline: raw 3M forwards. Scenario columns subtract k bp per year of horizon, an
illustration of how much a term premium could move the reading, not an estimate.

In [5]:
path = forwards.term_premium_scenarios(forwards.forward_path(p))
path.round(3)

,start_years,end_years,forward,expected_k0,expected_k10,expected_k20
0,0.00,0.25,2.563,2.563,2.563,2.563
1,0.25,0.50,2.949,2.949,2.924,2.899
2,0.50,0.75,3.193,3.193,3.143,3.093
3,0.75,1.00,3.339,3.339,3.264,3.189
4,1.00,1.25,3.420,3.420,3.320,3.220
5,1.25,1.50,3.459,3.459,3.334,3.209
6,1.50,1.75,3.472,3.472,3.322,3.172
7,1.75,2.00,3.470,3.470,3.295,3.120
8,2.00,2.25,3.461,3.461,3.261,3.061


## 5. PCA: level, slope, curvature

In [6]:
fit = pca.fit_pca(spot)
display(fit.explained.mul(100).round(1).rename("% of variance"))
display(fit.loadings.round(2))
{name: pca.sign_changes(fit.loadings[name].to_numpy()) for name in fit.loadings}

level        85.0
slope        10.6
curvature     2.7
Name: % of variance, dtype: float64

,level,slope,curvature
1.0,0.22,-0.42,-0.45
2.0,0.32,-0.45,-0.27
3.0,0.35,-0.36,-0.01
5.0,0.37,-0.16,0.32
7.0,0.37,-0.01,0.42
10.0,0.36,0.15,0.34
15.0,0.35,0.29,0.05
20.0,0.34,0.38,-0.22
30.0,0.30,0.47,-0.53


{'level': 0, 'slope': 1, 'curvature': 2}

Curve-shape anomaly (residual after three factors). On a fitted Svensson curve this is
**not** a tradable rich/cheap signal; it only shows where the fitted shape is unusual.

In [7]:
pca.anomaly_table(spot, fit).round(2)

,residual_bp,z,flag
1.0,-7.39,-0.56,
2.0,2.00,0.35,
3.0,4.54,0.57,
5.0,3.15,1.17,
7.0,0.33,0.12,
10.0,-2.94,-0.61,
15.0,-4.80,-1.26,
20.0,-2.91,-1.50,
30.0,6.14,1.27,


## 6. Everything the note is allowed to say

In [8]:
res = build.run()
pd.Series({k: v for k, v in res.metrics.items() if not isinstance(v, list)}).to_frame("value")

,value
as_of,2026-09-28
aaa_2y_pct,3.23
aaa_2y_chg_1w_bp,9
aaa_2y_chg_1m_bp,43
aaa_5y_pct,3.38
...,...
real_ea_10y_minus_spf_lt_pct,1.59
real_us_10y_minus_cpi_pct,1.46
real_us_tips_10y_pct,2.83
us_breakeven_10y_pct,2.34
